# Session 2: Broadcasting & Reshaping (2 hours)

**Goal:** Use one ML-style task (center features) to build intuition for broadcasting. Then predict shapes and apply the same idea to new sizes.

| Section | Time | Focus |
|---------|------|-------|
| Anchor case 1 | 40 min | Center each feature (subtract column mean) |
| Anchor case 2 | 35 min | Row normalization (each row sums to 1) |
| Anchor case 3 | 35 min | Outer product via broadcasting |
| Concept + predict | 25 min | Broadcasting rules, predict-before-run |
| Reshaping | 30 min | Essential reshape/expand_dims for ML |
| Visualization | 35 min | Plot broadcasting |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

---
## Anchor case: Center each feature (subtract per-column mean) [EASY]

**Task:** We have a (100, 3) matrix of features. Subtract the mean of **each column** so each feature has mean 0. This is standard preprocessing in ML.

**Why broadcasting?** `X.mean(axis=0)` gives shape (3,). To subtract it from every row of (100, 3), we need (100, 3) − (3,). NumPy broadcasts (3,) to (1, 3) then stretches to (100, 3). Using `keepdims=True` gives (1, 3) explicitly.

### Worked example

In [ ]:
np.random.seed(42)
X = np.random.randn(100, 3)  # 100 examples, 3 features
print(f"X.shape = {X.shape}")
print(f"Column means before: {X.mean(axis=0)}")

X_centered = X - X.mean(axis=0)
print(f"Column means after:  {X_centered.mean(axis=0)}")
print("(Should be ~[0,0,0])")

### Your turn [EASY]

1. Center a **(50, 5)** matrix (subtract each column mean). 2. Add a **(3,)** bias to every row of **(100, 3)** (result shape (100, 3)).

In [ ]:
np.random.seed(0)
A = np.random.randn(50, 5)
A_centered = ...  # YOUR CODE
print(f"A_centered column means: {A_centered.mean(axis=0)}")

B = np.random.randn(100, 3)
bias = np.array([0.1, -0.2, 0.0])
B_plus_bias = ...  # YOUR CODE
print(f"B_plus_bias.shape = {B_plus_bias.shape}")

<details>
<summary>💡 Click to see solution</summary>

```python
A_centered = A - A.mean(axis=0)
B_plus_bias = B + bias
```
</details>


### Sensemaking

**When would you use `reshape(-1, 1)` vs `reshape(1, -1)` in a regression?** What cues would you look for?

_Your answer: e.g. If y is (n,) and we need (n,1) to broadcast with a matrix, use reshape(-1,1). For a single row to broadcast across rows, use reshape(1,-1)._

---
## Anchor case 2: Row normalization — each row sums to 1 [MEDIUM]

**Task:** We have a matrix (n, d) of **scores** (e.g. logits). Make each **row** sum to 1 by dividing each row by its sum. This is like a softmax without the exp (or a simple row-stochastic matrix). Result shape (n, d).

**Why broadcasting?** Row sums have shape (n,). To divide (n, d) by (n,), we need (n,) to become (n, 1) so it broadcasts across columns.

### Worked example

In [ ]:
scores = np.array([[1.0, 2.0, 1.0], [0.5, 0.5, 1.0]])  # (2, 3)
row_sums = scores.sum(axis=1)   # (2,)
row_sums_2d = row_sums[:, np.newaxis]  # (2, 1) for broadcasting
normalized = scores / row_sums_2d
print(scores)
print("Row sums:", row_sums)
print(normalized)
print("Row sums after:", normalized.sum(axis=1))

### Your turn [MEDIUM]

Given a (5, 4) matrix of positive numbers, make each row sum to 1. Verify with `normalized.sum(axis=1)`.

In [ ]:
np.random.seed(0)
A = np.random.rand(5, 4) + 0.1
# YOUR CODE: row_normalized so each row sums to 1
row_normalized = ...
print(row_normalized.sum(axis=1))  # should be [1,1,1,1,1]

<details>
<summary>💡 Click to see solution</summary>

```python
row_normalized = A / A.sum(axis=1, keepdims=True)
```
</details>


### Sensemaking

**Why do we need `keepdims=True` (or [:, np.newaxis]) here instead of dividing by a 1D array of shape (n,)?**

_Your answer: (n, d) / (n,) would try to align trailing dims: d vs n, which don't match. (n, d) / (n, 1) broadcasts (n,1) to (n,d)._


---
## Anchor case 3: Outer product via broadcasting [MEDIUM]

**Task:** Given two vectors `a` (length 3) and `b` (length 4), compute the **outer product** matrix of shape (3, 4) where result[i,j] = a[i] * b[j], **without** using `np.outer`. Use only reshaping and broadcasting (e.g. `a[:, np.newaxis] * b[np.newaxis, :]`).

### Worked example

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])
# (3,) and (4,) → (3, 1) * (1, 4) → (3, 4)
outer = a[:, np.newaxis] * b[np.newaxis, :]
print(outer)
print("Match np.outer:", np.allclose(outer, np.outer(a, b)))

### Your turn [MEDIUM]

Compute the outer product of `u = np.array([1, 2, 3, 4])` and `v = np.array([-1, 0, 1])` using broadcasting. Check shape (4, 3).

In [ ]:
u = np.array([1, 2, 3, 4])
v = np.array([-1, 0, 1])
# YOUR CODE: outer_uv shape (4, 3)
outer_uv = ...
print(outer_uv)
assert outer_uv.shape == (4, 3)

<details>
<summary>💡 Click to see solution</summary>

```python
outer_uv = u[:, np.newaxis] * v[np.newaxis, :]
```
</details>


### Sensemaking

**In ML, where might a (n, d) * (d, k) or outer-like broadcast appear?** (One idea: attention scores, or pairwise distances.)

_Your answer: e.g. Pairwise distances: (n,1,d) - (1,m,d) then sum over last dim gives (n,m). Or scores between n queries and m keys._

### Concept: The three broadcasting rules

1. **Align right:** Shapes compared from trailing dimensions. 2. **Match or 1:** Equal or one is 1. 3. **Stretch:** Dimension 1 is stretched. **Predict before you run:**

In [ ]:
# Pair 1: (2,3) + (3,) — common in ML (matrix + row bias)
a = np.ones((2, 3))
b = np.ones((3,))
result = a + b
print(f"Pair 1: {a.shape} + {b.shape} = {result.shape}")

In [ ]:
# Pair 2
a = np.ones((4, 1))     # Shape: (4, 1)
b = np.ones((1, 3))     # Shape: (1, 3)
# Your prediction: ___

result = a + b
print(f"Pair 2: {a.shape} + {b.shape} = {result.shape}")

In [ ]:
# Pair 3
a = np.ones((3, 4, 5))  # Shape: (3, 4, 5)
b = np.ones((4, 1))     # Shape: (4, 1)
# Your prediction: ___

result = a + b
print(f"Pair 3: {a.shape} + {b.shape} = {result.shape}")

### Exercise 2: Practical Broadcasting

Use broadcasting to:
1. Subtract the column mean from each column of a matrix
2. Compute an outer product using broadcasting instead of `np.outer`
3. Compute pairwise Euclidean distances between two sets of points

In [ ]:
# 2a. Subtract column mean
np.random.seed(42)
X = np.random.randint(0, 10, (4, 3)).astype(float)
print("Original X:")
print(X)

# YOUR CODE HERE: center each column (subtract its mean)
X_centered = ...

print("\nCentered X:")
print(X_centered)
print(f"Column means (should be ~0): {X_centered.mean(axis=0)}")

<details>
<summary>💡 Click to see solution</summary>

```python
X_centered = X - X.mean(axis=0)
```
</details>


In [ ]:
# 2b. Outer product via broadcasting
a = np.array([1, 2, 3])
b = np.array([4, 5, 6, 7])

# YOUR CODE HERE: compute outer product using broadcasting (no np.outer!)
outer = ...

# Verify
print(f"Broadcasting outer:\n{outer}")
print(f"np.outer:\n{np.outer(a, b)}")
print(f"Match: {np.allclose(outer, np.outer(a, b))}")

<details>
<summary>💡 Click to see solution</summary>

```python
outer = a[:, np.newaxis] * b[np.newaxis, :]
# Or: outer = a.reshape(-1, 1) * b.reshape(1, -1)
```
</details>


In [ ]:
# 2c. Pairwise distances
# Given two sets of 2D points, compute the distance from every point in A to every point in B
np.random.seed(0)
A = np.random.rand(5, 2)   # 5 points in 2D
B = np.random.rand(3, 2)   # 3 points in 2D

# YOUR CODE HERE: compute (5, 3) distance matrix using broadcasting
# Hint: reshape A to (5, 1, 2) and B to (1, 3, 2), then compute
distances = ...

print(f"Distance matrix shape: {distances.shape}")
print(distances)

<details>
<summary>💡 Click to see solution</summary>

```python
# Reshape A to (5, 1, 2) and B to (1, 3, 2)
diff = A[:, np.newaxis, :] - B[np.newaxis, :, :]  # (5, 3, 2)
distances = np.sqrt((diff ** 2).sum(axis=2))
```
</details>


---
## Task 2.2: Reshaping Mastery (45 min)

### Exercise 3: Core Reshaping Operations

In [ ]:
arr = np.arange(24)
print(f"Original: shape={arr.shape}")
print(arr)

# Reshape into different configurations
r1 = arr.reshape(4, 6)
r2 = arr.reshape(2, 3, 4)
r3 = arr.reshape(6, -1)       # -1 lets NumPy infer the dimension
r4 = arr.reshape(-1, 2, 3)

print(f"\n(4, 6):\n{r1}")
print(f"\n(2, 3, 4):\n{r2}")
print(f"\n(6, -1) -> {r3.shape}:\n{r3}")
print(f"\n(-1, 2, 3) -> {r4.shape}:\n{r4}")

### Exercise 4: Transpose

In [ ]:
A = np.arange(12).reshape(3, 4)
print(f"A shape: {A.shape}")
print(A)

# Transpose with .T
print(f"\nA.T shape: {A.T.shape}")
print(A.T)

# For 3D arrays, you can specify axis order
B = np.arange(24).reshape(2, 3, 4)
print(f"\nB shape: {B.shape}")

# Swap axes 0 and 2
B_t = B.transpose(2, 1, 0)
print(f"B.transpose(2,1,0) shape: {B_t.shape}")

### Exercise 5: expand_dims, squeeze, newaxis

In [ ]:
v = np.array([1, 2, 3])  # Shape: (3,)

# Three ways to make it a column vector (3, 1):
col1 = v[:, np.newaxis]
col2 = np.expand_dims(v, axis=1)
col3 = v.reshape(-1, 1)

print(f"v.shape: {v.shape}")
print(f"col1.shape: {col1.shape}  (np.newaxis)")
print(f"col2.shape: {col2.shape}  (expand_dims)")
print(f"col3.shape: {col3.shape}  (reshape)")

# Squeeze removes dimensions of size 1
bloated = np.zeros((1, 3, 1, 5, 1))
print(f"\nBloated shape: {bloated.shape}")
print(f"Squeezed shape: {np.squeeze(bloated).shape}")

### Exercise 6: Stacking and Concatenating

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Stack vertically, horizontally, and depth-wise
print(f"vstack: {np.vstack([a, b]).shape}")
print(np.vstack([a, b]))

print(f"\nhstack: {np.hstack([a, b]).shape}")
print(np.hstack([a, b]))

print(f"\nstack(axis=0): {np.stack([a, b], axis=0).shape}")
print(np.stack([a, b], axis=0))

print(f"\nstack(axis=1): {np.stack([a, b], axis=1).shape}")
print(np.stack([a, b], axis=1))

### Exercise 7: ravel vs flatten

In [ ]:
A = np.arange(6).reshape(2, 3)

# ravel returns a VIEW when possible (shares memory)
r = A.ravel()
r[0] = 999
print(f"After ravel modification, A[0,0] = {A[0,0]}")

# Reset
A = np.arange(6).reshape(2, 3)

# flatten always returns a COPY
f = A.flatten()
f[0] = 999
print(f"After flatten modification, A[0,0] = {A[0,0]}")

### Exercise 8: Reshaping Challenge

Given the image-like array below, perform the requested transformations.

In [ ]:
# Simulate a batch of 2 RGB images, each 4x4 pixels
# Shape: (batch, height, width, channels) = (2, 4, 4, 3)
images = np.random.randint(0, 256, (2, 4, 4, 3))

# a) Convert to channels-first format: (batch, channels, height, width)
channels_first = ...  # YOUR CODE
print(f"Channels first: {channels_first.shape}")

# b) Flatten each image to a 1D vector: (batch, height*width*channels)
flattened = ...  # YOUR CODE
print(f"Flattened: {flattened.shape}")

# c) Extract just the red channel from all images: (batch, height, width)
red_channel = ...  # YOUR CODE
print(f"Red channel: {red_channel.shape}")

<details>
<summary>💡 Click to see solution</summary>

```python
# a) Channels first: (batch, channels, height, width)
channels_first = np.transpose(images, (0, 3, 1, 2))

# b) Flatten each image
flattened = images.reshape(images.shape[0], -1)

# c) Red channel (index 0 in last dim)
red_channel = images[:, :, :, 0]
```
</details>


### Checkpoint

Can you predict the output shape of `np.random.randn(3, 4, 5) + np.ones((4, 1))`?

**Your prediction:** ___

Run the cell below to verify:

In [ ]:
result = np.random.randn(3, 4, 5) + np.ones((4, 1))
print(f"Result shape: {result.shape}")

---
## Task 2.3: Visualization (30 min)

Create a visual explanation of broadcasting.

In [ ]:
def annotate_heatmap(ax, data, fmt=".0f"):
    """Add text annotations to a heatmap."""
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, format(data[i, j], fmt),
                    ha="center", va="center", color="black", fontsize=12)


def visualize_broadcasting():
    a = np.arange(6).reshape(2, 3)
    b = np.array([10, 20, 30])
    c = a + b

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    # Matrix A
    axes[0].imshow(a, cmap='Blues', aspect='auto')
    annotate_heatmap(axes[0], a)
    axes[0].set_title(f'A  {a.shape}', fontsize=14)
    axes[0].set_xticks(range(3))
    axes[0].set_yticks(range(2))

    # Plus sign
    axes[1].text(0.5, 0.5, '+', fontsize=40, ha='center', va='center')
    axes[1].axis('off')

    # Vector b (broadcast to 2D for display)
    b_display = np.tile(b, (2, 1))
    axes[2].imshow(b_display, cmap='Oranges', aspect='auto')
    annotate_heatmap(axes[2], b_display)
    axes[2].set_title(f'B  {b.shape}\n(broadcast to {b_display.shape})', fontsize=14)
    axes[2].set_xticks(range(3))
    axes[2].set_yticks(range(2))

    # Result
    axes[3].imshow(c, cmap='Greens', aspect='auto')
    annotate_heatmap(axes[3], c)
    axes[3].set_title(f'A + B  {c.shape}', fontsize=14)
    axes[3].set_xticks(range(3))
    axes[3].set_yticks(range(2))

    plt.suptitle('Broadcasting: (2,3) + (3,) = (2,3)', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig('images/broadcasting_visual.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to images/broadcasting_visual.png")

visualize_broadcasting()

In [ ]:
def visualize_broadcasting_2d():
    """Visualize (4,1) + (1,3) = (4,3) broadcasting."""
    col = np.array([[1], [2], [3], [4]])    # (4, 1)
    row = np.array([[10, 20, 30]])           # (1, 3)
    result = col + row                       # (4, 3)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5),
                              gridspec_kw={'width_ratios': [1, 0.3, 3, 3]})

    axes[0].imshow(col, cmap='Blues', aspect='auto')
    annotate_heatmap(axes[0], col)
    axes[0].set_title(f'Column {col.shape}', fontsize=13)
    axes[0].set_xticks([0])
    axes[0].set_yticks(range(4))

    axes[1].text(0.5, 0.5, '+', fontsize=36, ha='center', va='center')
    axes[1].axis('off')

    row_display = np.tile(row, (4, 1))
    axes[2].imshow(row_display, cmap='Oranges', aspect='auto')
    annotate_heatmap(axes[2], row_display)
    axes[2].set_title(f'Row {row.shape}\n(broadcast)', fontsize=13)
    axes[2].set_xticks(range(3))
    axes[2].set_yticks(range(4))

    axes[3].imshow(result, cmap='Greens', aspect='auto')
    annotate_heatmap(axes[3], result)
    axes[3].set_title(f'Result {result.shape}', fontsize=13)
    axes[3].set_xticks(range(3))
    axes[3].set_yticks(range(4))

    plt.suptitle('Broadcasting: (4,1) + (1,3) = (4,3)', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig('images/broadcasting_2d_visual.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_broadcasting_2d()

---
## Session 2 Reflection

**Can you now predict output shapes without running code?**

_Your answer..._

**What's the difference between `ravel` and `flatten`?**

_Your answer..._

**Connection:** Centering columns (subtract mean) is the same idea we'll use when we normalise features in linear regression (Session 5).